# Module 19: Bluetooth Remote

## Mission Briefing

In Module 15, your Pico learned to connect to **WiFi** — talking to the internet through a router. But what if you want to control your Pico directly from your phone, with **no internet needed**?

Today your Pico becomes a **Bluetooth device** — just like wireless earbuds or a game controller. You'll send commands from your phone to control an LED and buzzer!

```
    Your Phone                         Your Pico 2W
   ┌──────────┐                      ┌──────────────┐
   │          │   ~ Bluetooth ~      │              │
   │  "on"   ─┼──────))))──────────>─┼─── LED ON!   │
   │  "off"  ─┼──────))))──────────>─┼─── LED OFF!  │
   │  "beep" ─┼──────))))──────────>─┼─── BEEP!     │
   │          │    No WiFi needed!   │              │
   └──────────┘                      └──────────────┘
```

No router, no internet, no wires — just your phone and your Pico, talking directly!

---
## Science Spotlight: WiFi vs Bluetooth

WiFi connects to the internet. Bluetooth connects directly to nearby devices — like wireless earbuds or a game controller. Today your Pico becomes a Bluetooth device!

We're using **BLE** — **Bluetooth Low Energy** — a special version designed for small, battery-powered gadgets. It uses very little power, so devices like fitness trackers can run for months on a tiny battery.

*Your instructor will explain how BLE advertising works and how data is organized into services and characteristics — like a restaurant menu!*

---
## Components

```
   ┌─────────────────┐
   │  Pico 2W        │   Already has BLE built in — no extra radio needed!
   └─────────────────┘

   ┌───┐
   │LED│  Your trusty LED (any color)
   └───┘

   ┌──────┐
   │ 220Ω │  Resistor to protect the LED
   └──────┘

   ┌──────┐
   │Buzzer│  From Module 8 — for beep commands
   └──────┘

   Jumper wires + breadboard
```

You'll also need a **phone app** to send Bluetooth commands:
- **Android:** "Serial Bluetooth Terminal" (free on Play Store)
- **iPhone/iPad:** "LightBlue" (free on App Store)

Ask your instructor which app to install!

---
## Wiring: LED + Buzzer

We're wiring an LED and a buzzer so we can control them wirelessly.

### Circuit Diagram

```
   Pico 2W
   ┌────────────────────────┐
   │                        │
   │  GP15 (pin 20) ───────┼──── 220Ω ──── LED (+) ──── LED (-) ──── GND
   │                        │
   │  GP14 (pin 19) ───────┼──── Buzzer (+) ──── Buzzer (-) ──── GND
   │                        │
   └────────────────────────┘
```

### Step-by-Step

1. **LED:** Connect GP15 → 220Ω resistor → LED long leg (anode). Connect LED short leg (cathode) → GND.
2. **Buzzer:** Connect GP14 → buzzer positive (+) leg. Connect buzzer negative (-) leg → GND.
3. Plug in your USB cable.

**Note:** The Bluetooth radio is built into the Pico 2W — no extra wiring needed for wireless!

---
## Understanding BLE: The Big Picture

Before we code, let's understand how BLE works:

```
   Step 1: ADVERTISE              Step 2: CONNECT              Step 3: COMMUNICATE
   
   Pico: "Hey! I'm here!          Phone: "I see you!           Phone ──> Pico
          My name is               Let me connect."             "on"
          PicoRemote!"                                          "off"
          ))))                     Phone ←──── Pico             "beep"
          ))))                     Connected!
          ))))                                                  Pico controls
                                                                LED + buzzer!
```

1. **Advertise** — The Pico shouts "I'm here!" over Bluetooth (like a shop sign)
2. **Connect** — Your phone sees the Pico and connects to it
3. **Communicate** — Your phone sends text commands, the Pico reads them and acts

The BLE setup code is complex, so we've provided most of it for you. Your job is to focus on the **fun part** — deciding what commands do!

---
## Code Along: BLE Setup (Boilerplate)

This cell sets up all the BLE plumbing. Most of it is provided for you — you only need to fill in a few blanks.

**Don't worry about understanding every line!** The important thing is that this code makes your Pico appear as a Bluetooth device that can receive text messages.

Fill in the blanks:

In [ ]:
import bluetooth
import struct
import time
from machine import Pin, PWM
from micropython import const

# --- Hardware Setup ---
led = Pin(_____, Pin.OUT)          # Which GPIO pin is the LED on?
buzzer = PWM(Pin(_____))           # Which GPIO pin is the buzzer on?
buzzer.duty_u16(0)                 # Start with buzzer silent

# --- BLE Constants (don't change these) ---
_IRQ_CENTRAL_CONNECT = const(1)
_IRQ_CENTRAL_DISCONNECT = const(2)
_IRQ_GATTS_WRITE = const(3)

# UART Service UUID — a standard Bluetooth "language" for sending text
_UART_UUID = bluetooth.UUID("6E400001-B5A3-F393-E0A9-E50E24DCCA9E")
_UART_TX = (
    bluetooth.UUID("6E400003-B5A3-F393-E0A9-E50E24DCCA9E"),
    bluetooth.FLAG_NOTIFY,
)
_UART_RX = (
    bluetooth.UUID("6E400002-B5A3-F393-E0A9-E50E24DCCA9E"),
    bluetooth.FLAG_WRITE,
)
_UART_SERVICE = (_UART_UUID, (_UART_TX, _UART_RX))

# --- BLE Class ---
class BLERemote:
    def __init__(self, name="_____"):     # Give your device a name! (e.g., "PicoRemote")
        self._ble = bluetooth.BLE()
        self._ble.active(True)
        self._ble.irq(self._irq)
        ((self._tx_handle, self._rx_handle),) = self._ble.gatts_register_services((_UART_SERVICE,))
        self._connections = set()
        self._rx_buffer = b""
        self._name = name
        self._advertise()
        print("BLE device '", name, "' is advertising...")

    def _irq(self, event, data):
        if event == _IRQ_CENTRAL_CONNECT:
            conn_handle, _, _ = data
            self._connections.add(conn_handle)
            print("Phone connected!")
        elif event == _IRQ_CENTRAL_DISCONNECT:
            conn_handle, _, _ = data
            self._connections.discard(conn_handle)
            print("Phone disconnected.")
            self._advertise()    # Start advertising again
        elif event == _IRQ_GATTS_WRITE:
            conn_handle, value_handle = data
            value = self._ble.gatts_read(value_handle)
            self._rx_buffer += value

    def _advertise(self):
        name_bytes = self._name.encode()
        # Build advertising payload: flags + complete name
        payload = bytearray()
        # Flags
        payload += struct.pack("BBB", 2, 0x01, 0x06)
        # Complete local name
        payload += struct.pack("BB", len(name_bytes) + 1, 0x09) + name_bytes
        self._ble.gap_advertise(250000, adv_data=payload)

    def is_connected(self):
        return len(self._connections) > 0

    def read(self):
        """Read any data received from the phone."""
        if not self._rx_buffer:
            return None
        data = self._rx_buffer
        self._rx_buffer = b""
        return data.decode().strip()

    def send(self, text):
        """Send text back to the phone."""
        for conn in self._connections:
            self._ble.gatts_notify(conn, self._tx_handle, text.encode())

print("BLE setup code loaded!")

If you see "BLE setup code loaded!" — great! This cell just defines the class. The next cell will actually start advertising.

---
## Code Along: Start Advertising

Now let's create our BLE device and start advertising. Once this runs, your Pico will appear on your phone's Bluetooth scanner!

Fill in the blank:

In [ ]:
# Create the BLE remote — this starts advertising
remote = _____()             # What class did we define in the previous cell?

print("Waiting for a phone to connect...")
print("Open your Bluetooth app and look for the device!")

# Wait for a connection
while not remote.is_connected():
    time.sleep(0.5)
    print(".", end="")

print("\nConnected! Ready to receive commands.")

### How to Connect from Your Phone

**Android (Serial Bluetooth Terminal):**
1. Open the app
2. Tap the menu (three lines) → Devices
3. Tap "Bluetooth LE" tab at the top
4. Tap SCAN — find "PicoRemote" (or whatever name you chose)
5. Tap on it to connect
6. You should see "Phone connected!" in Thonny

**iPhone/iPad (LightBlue):**
1. Open LightBlue
2. It auto-scans — find "PicoRemote" in the list
3. Tap on it to connect
4. Find the UART RX characteristic (UUID ending in 0002)
5. Tap "Write new value" to send text

Once connected, you'll see the message in Thonny. **Don't disconnect yet — we need the connection for the next step!**

---
## Code Along: Receive and Parse Commands

Now the exciting part! We'll listen for text commands from the phone and use them to control the LED and buzzer.

This is where YOU get to decide what happens for each command!

Fill in the blanks:

In [ ]:
# Make sure you ran the BLE setup cell first!
# And that your phone is connected.

def do_beep():
    """Play a short beep on the buzzer."""
    buzzer.freq(1000)
    buzzer.duty_u16(32768)
    time.sleep(0.2)
    buzzer.duty_u16(0)

print("Listening for commands...")
print("Send 'on', 'off', or 'beep' from your phone.")

while True:
    if remote.is_connected():
        command = remote.read()
        
        if command:                           # Did we receive something?
            command = command.lower()          # Make it lowercase
            print("Received:", command)
            
            if command == "on":
                led._____(___)               # Turn the LED ON — what method? what value?
                remote.send("LED is ON!")
                
            elif command == "_____":          # What command should turn the LED off?
                led.value(0)
                remote.send("LED is OFF!")
                
            elif command == "_____":          # What command should make the buzzer beep?
                do_beep()
                remote.send("BEEP!")
                
            else:
                print("Unknown command:", command)
                remote.send("Unknown: " + command)
    
    time.sleep(0.1)                           # Small delay to avoid busy-waiting

### Try It!

From your phone app, type and send these commands:
- **on** — LED should light up
- **off** — LED should turn off
- **beep** — buzzer should make a short sound
- Try typing something random — what happens?

**You just built a wireless remote control!** Your phone sends text over Bluetooth, and your Pico reads it and acts on it. No internet needed!

---
## Code Along: Adding More Commands

Let's make our remote more powerful by adding a **blink** command. When the phone sends "blink", the LED should blink 3 times.

Fill in the blanks:

In [ ]:
def do_beep():
    buzzer.freq(1000)
    buzzer.duty_u16(32768)
    time.sleep(0.2)
    buzzer.duty_u16(0)

def do_blink(times):
    """Blink the LED a given number of times."""
    for i in range(_____):
        led.value(_____)
        time.sleep(0.3)
        led.value(_____)
        time.sleep(0.3)

print("Listening for commands...")
print("Commands: on, off, beep, blink")

while True:
    if remote.is_connected():
        command = remote.read()
        
        if command:
            command = command.lower()
            print("Received:", command)
            
            if command == "on":
                led.value(1)
                remote.send("LED is ON!")
                
            elif command == "off":
                led.value(0)
                remote.send("LED is OFF!")
                
            elif command == "beep":
                do_beep()
                remote.send("BEEP!")
                
            elif command == "_____":         # What command triggers blinking?
                do_blink(3)
                remote.send("Blinked 3 times!")
                
            else:
                remote.send("Unknown: " + command)
    
    time.sleep(0.1)

---
## Experiments

Try these modifications:

1. **Range test:** Stay connected and walk away from your Pico. How far can you go before it disconnects? Try different rooms. BLE range is typically about 10 meters indoors.

2. **Change the beep:** Modify `do_beep()` to play a different frequency. Try `buzzer.freq(2000)` for a higher pitch, or `buzzer.freq(500)` for a lower pitch.

3. **Longer blink:** Change `do_blink(3)` to `do_blink(5)`. Can you make the phone send a NUMBER to control how many times it blinks?

4. **Status check:** Add a command "status" that sends back whether the LED is currently on or off. Hint: use `led.value()` to check.

5. **Rename your device:** Change the name in `BLERemote(name="...")` to something fun. Reconnect from your phone — does the new name show up?

---
## Challenge: SOS and Custom Patterns

Add these commands to your Bluetooth remote:

### 1. SOS Command
When the phone sends "sos", blink the LED in Morse code:
- S = three short blinks (dot dot dot)
- O = three long blinks (dash dash dash)
- S = three short blinks (dot dot dot)

Short blink = 0.1 seconds on, long blink = 0.4 seconds on.

### 2. Alarm Command
When the phone sends "alarm", make the buzzer play a rising tone (like a siren) while the LED blinks rapidly.

### 3. Custom Pattern
Invent your OWN command! Some ideas:
- "party" — LED blinks fast while buzzer plays a tune
- "heartbeat" — two quick blinks, pause, repeat
- "countdown" — blink 5 times, getting faster, then beep

**Starter code for SOS:**

```python
def do_sos():
    # S = three short blinks
    for i in range(3):
        led.value(1)
        time.sleep(0.1)    # Short!
        led.value(0)
        time.sleep(0.1)
    time.sleep(0.3)        # Pause between letters
    # O = three long blinks
    # ... your code here!
    # S = three short blinks again
    # ... your code here!
```

Ask your instructor for hints if you get stuck!

In [ ]:
# Your challenge code here!


---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Bluetooth** | A wireless technology for connecting devices directly, without a router |
| **BLE** | Bluetooth Low Energy — a power-efficient version of Bluetooth for small devices |
| **Advertising** | When a BLE device broadcasts "I'm here!" so other devices can find it |
| **Connecting** | When a phone pairs with the BLE device to exchange data |
| **UART** | Universal Asynchronous Receiver/Transmitter — a way to send text data over BLE |
| **Service** | A category of BLE data (like a section on a restaurant menu) |
| **Characteristic** | A specific piece of BLE data within a service (like an item on the menu) |
| **Command parsing** | Reading text input and deciding what action to take based on it |

---
*Circuit Explorers — Module 19: Bluetooth Remote*